In [ ]:

import pandas as pd

import warnings

warnings.filterwarnings('ignore')

#加载数据
print("="*60)
print("选题A12：代谢综合征组分重构 - 严格排除版（CDS 2020标准）")
print("="*60)

data_path = "../数据/Dataset for diabetes research.csv"

df = pd.read_csv(data_path, encoding='gbk', na_values=['', ' ', 'NA', 'NULL', 'null'])
df.columns = [col.lower() for col in df.columns]
gender_mapping = {1: 1, 2: 0}
df['gender'] = df['gender'].map(gender_mapping)

print("映射后性别分布：")
print(df['gender'].value_counts().sort_index())

print(f"原始数据形状: {df.shape}")
#所需变量列表
vars_to_check = ['waist1', 'gender', 'bpsys', 'bpdia', 'who1999hbp',
                 'fpg', 'pg2h', 'dm', 'tg', 'hdl']

#每个变量的缺失比例
missing_rates = df[vars_to_check].isna().mean().map(lambda x: f"{x:.2%}")
#输出结果
print("\n各组分指标缺失率：")
print(missing_rates)

#构建MS_CDS标签（CDS 2020标准）
def build_MS_CDS(df):
    df = df.copy()
    #腹型肥胖
    central_obesity = ((df['gender'] == 1) & (df['waist1'] >= 90)) | \
                      ((df['gender'] == 0) & (df['waist1'] >= 85))
    central_obesity = central_obesity.fillna(False)
    #高血压
    hypertension = ((df['bpsys'] >= 130) | (df['bpdia'] >= 85)) | (df['who1999hbp'] == 1)
    hypertension = hypertension.fillna(False)
    #高血糖
    hyperglycemia = ((df['fpg'] >= 6.1) | (df['pg2h'] >= 7.8) | (df['dm'] == 1))
    hyperglycemia = hyperglycemia.fillna(False)
    #高甘油三酯
    hypertrig = (df['tg'] >= 1.7)
    hypertrig = hypertrig.fillna(False)
    #低HDL
    low_hdl = (df['hdl'] < 1.04)
    low_hdl = low_hdl.fillna(False)

    #五个组分计数
    conditions = [central_obesity, hypertension, hyperglycemia, hypertrig, low_hdl]
    condition_count = sum(conditions)

    #诊断规则：5项中满足任意3项
    MS_CDS = (condition_count >= 3).astype(int)
    return MS_CDS

print("\n构建MS_CDS标签...")
df['ms_cds'] = build_MS_CDS(df)
print(f"MS患病率: {df['ms_cds'].mean():.4f} ({df['ms_cds'].sum()} 例)")
df = df.dropna(subset=['ms_cds'])
#严格排除所有与MS核心组分相关的特征以及冗余变量

exclude_features = [
    'var00001', 'no', 'data', 'eyedia', 'filter', 'gastudy', 'ganum',
    'waist1', 'waist2', 'wc9080', 'wc9085', 'wc8580',
    'bmi', 'weight', 'height', 'hip', 'whr', 'leg',
    'overweight', 'obesity', 'wgocobbmi', 'bmi25', 'bmi4class',
    'fat', 'fat2class', 'fat3class',
    'bpsys', 'bpdia', 'bp13085', 'who1999hbp',
    'fpg', 'glu', 'pg2h', 'hba1c', 'hplc', 'ga', 'dm',
    'pgclass3', 'pgclass6', 'dmnew3class', 'fpgover7', 'p2pgover11',
    'a1cover6575', 'dmhighrisk',
    'chol', 'tg', 'hdl', 'ldl', 'tgdis', 'dyshdlidfncep', 'jcdhdldis', 'cdsdyelip',
    'fins', 'fcp', 'ins2h', 'cp2h', 'homair', 'homaβ', 'isigutt',
    'hypertenhis', 'dyslipidehis', 'cdsms', 'dmhis',
    'hyper13',
    'hyper16',
    'hypo14',
    'dr', 'dryd', 'cataract', 'df', 'madpn', 'tfdpn', 'dn',
    'mi', 'hf', 'angina', 'arrhythmia', 'pad',
    'cerebralhe', 'cerebralin', 'tia', 'cap', 'rf', 'amputation',
    'blind', 'cancer', 'weightloss',
    'gfr', 'gfr5lev', 'gfr6levg5',
    'eugfrabacr012', 'eugfr90uacr01', 'eugfr60abacr012', 'uacrgfr',
    'thalussemiaand', 'thalussemiaor', 'thalussemiahplc',
    'alt.1',
    'age3group', 'agegroup',
    'altn',
    'tbiln',
    'highcr',
    'highua',
    'highumalb',
    'highacr',
    'acr3degree',
    'hb4per',
    'fib4n',
    'anemia',
    'smoking2',
    'drinking2',
    'astalt',
    'umaucr',
    'dmfamilyhistory',
    'rdwsd'
]
non_core_features = [c for c in df.columns if c not in exclude_features]
df[non_core_features].to_csv("../数据/1.标签构造后的数据.csv", index=False)

In [ ]:
df[non_core_features].shape

In [ ]:
df[non_core_features].columns.to_list()